# Lasso regression on the Extended Boston housing dataset

In [ ]:
# a lot of imports
%matplotlib inline
import sklearn
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score

print("Done")

**Ames housing Dataset**

In [ ]:
# --- 2. Load Ames dataset (real-world, not synthetic) ---
X, y = fetch_openml(name="house_prices", as_frame=True, return_X_y=True)

# --- 3. Basic preprocessing ---
# Separate numeric and categorical columns
num_cols = X.select_dtypes(include=["number"]).columns
cat_cols = X.select_dtypes(exclude=["number"]).columns
cat_cols


# In the following we normalize numerical data and one-hot encode categorical input values
featureMatrix = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
   ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
]).fit_transform(X)
featureMatrix.shape

# second preprocessing replacing NaN
featureMatrix = np.nan_to_num(featureMatrix, nan=0.0)
featureMatrix.shape

# --- 4. Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    featureMatrix, y, random_state=100
)

# --- 5. Models ---
lr = LinearRegression()
lr.fit(X_train, y_train)

print("Results without regularization")
print("Train R²:", r2_score(y_train, lr.predict(X_train)))
print("Test  R²:", r2_score(y_test,  lr.predict(X_test)))


<b>Lasso Regression - Introduction of L1-"Regularization"</b>

In [ ]:
# Lasso drives regularization to be coefficients to be exactly 0
from sklearn.linear_model import Lasso

lasso = Lasso().fit(X_train, y_train)  # no arguments means alpha = 1 
print("Training set score: {:.2f}".format(lasso.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lasso.score(X_test, y_test)))
print("Number of features used:", np.sum(lasso.coef_ != 0))

In [ ]:
lasso.alpha

In [ ]:
# we increase the default setting of "max_iter",
# otherwise the model would warn us that we should increase max_iter.
lassoA = Lasso(alpha=10, max_iter=10000).fit(X_train, y_train)
print("Training set score: {:.2f}".format(lassoA.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lassoA.score(X_test, y_test)))
print("Number of features used:", np.sum(lassoA.coef_ != 0))

In [ ]:
lassoB = Lasso(alpha=300, max_iter=100000).fit(X_train, y_train)
print("Training set score: {:.2f}".format(lassoB.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lassoB.score(X_test, y_test)))
print("Number of features used:", np.sum(lassoB.coef_ != 0))

**Comparing Coefficients of Lasso with different $\alpha$**

In [ ]:
plt.figure(figsize=(12, 12))
plt.plot(lasso.coef_[0:100], 's', label="Lasso alpha=1 (0.54 Test Score)")
plt.plot(lassoA.coef_[0:100], '^', label="Lasso alpha=0.01 (0.82 Test Score)")
plt.plot(lassoB.coef_[0:100], 'v', label="Lasso alpha=0.0001 (0.87 Test Score)")


plt.ylim(-40000, 40000)
plt.xlabel("Coefficient index")
plt.ylabel("Coefficient magnitude")
plt.legend()
plt.show()

**Lasso versus Ridge - Visualization of Coefficient Range**

In [ ]:
from sklearn.linear_model import Ridge
ridge100 = Ridge(alpha=100).fit(X_train, y_train) # best ridge run from last notebook

plt.figure(figsize=(12, 12))
plt.plot(lassoB.coef_, '^', label="Lasso alpha=0.01 (0.87 Test Score)")
plt.plot(ridge100.coef_, 'o', label="Ridge alpha=0.1 (0.87 Test Score)")
plt.ylim(-1000, 1000)
plt.xlabel("Coefficient index")
plt.ylabel("Coefficient magnitude")
plt.legend()
plt.show()